Load And Use Finetuned Model

In [ ]:
# 打印关键依赖库的版本，便于复现环境（tiktoken 负责分词，torch 是深度学习框架）
from importlib.metadata import version

pkgs = [
    "tiktoken",    # Tokenizer（GPT-2 使用的 BPE 分词器）
    "torch",       # Deep learning library
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
from pathlib import Path

# 检查第7章指令微调后保存的权重文件是否存在；不存在则提示先运行 ch07.ipynb 生成
finetuned_model_path = Path("gpt2-medium355M-sft.pth")
if not finetuned_model_path.exists():
    print(
        f"Could not find '{finetuned_model_path}'.\n"
        "Run the `ch07.ipynb` notebook to finetune and save the finetuned model."
    )
from previous_chapters import GPTModel  # 复用前几章实现的 GPT 模型类


# GPT-2 各规模通用的基础配置（词表大小、上下文长度、dropout、是否给 QKV 加偏置）
BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size（GPT-2 词表大小）
    "context_length": 1024,  # Context length（最大上下文长度）
    "drop_rate": 0.0,        # Dropout rate（推理阶段设 0）
    "qkv_bias": True         # Query-key-value bias（为匹配 OpenAI 官方权重结构需为 True）
}

# 不同规模 GPT-2 的结构差异：嵌入维度 / 层数 / 注意力头数
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"  # 选择与微调时一致的规模（355M）

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])  # 把所选规模的结构参数并入基础配置

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")  # 从名字里提取 "355M" 这样的规模字符串
model = GPTModel(BASE_CONFIG)  # 按配置实例化模型（结构须与保存权重时完全一致，否则 load 会失败）
import torch

# 载入微调后的权重；map_location=cpu 便于无 GPU 也能加载，weights_only=True 更安全（只反序列化张量）
model.load_state_dict(torch.load(
    "gpt2-medium355M-sft.pth",
    map_location=torch.device("cpu"),
    weights_only=True
))
model.eval();  # 切到评估模式（关闭 dropout 等），推理必需；行尾分号抑制 Jupyter 的对象打印输出
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")  # GPT-2 的 BPE 分词器
# 用与训练时一致的 Alpaca 风格指令模板构造 prompt（Instruction 段 + 具体任务）
prompt = """Below is an instruction that describes a task. Write a response
that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'
"""
from previous_chapters import (
    generate,             # 采样/贪心解码的文本生成函数
    text_to_token_ids,    # 文本 -> token id 张量
    token_ids_to_text     # token id 张量 -> 文本
)

def extract_response(response_text, input_text):
    # 模型输出里包含了原始 prompt，这里切掉 prompt 部分、去掉 "### Response:" 标记，只保留真正的回答
    return response_text[len(input_text):].replace("### Response:", "").strip()

torch.manual_seed(123)  # 固定随机种子，让生成结果可复现

# 生成回复：最多再生成 35 个新 token，遇到 EOS(50256) 提前停止
token_ids = generate(
    model=model,
    idx=text_to_token_ids(prompt, tokenizer),  # 形状 (1, prompt_len)
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256
)

response = token_ids_to_text(token_ids, tokenizer)   # 解码成文本
response = extract_response(response, prompt)          # 抽取出纯回答部分
print(response)